# HMST-v2 — Example Notebook

This notebook shows the **minimum code** needed to train and evaluate any model in the HMST package.
Every implementation detail (training loop, dataset, metrics) lives inside the `hmst` package —
the notebook reads like English.

```
pip install git+https://github.com/<your-org>/hmst.git
```

## 1. Imports

In [ ]:
import torch
import numpy as np
import pandas as pd

# ── All model & training APIs come from the hmst package ──────────────────
from hmst.model  import HMSTv2, AllGridLSTM, AllGridConvLSTM, AllGridDLinear, AllGridPatchTST, EachGridLSTM
from hmst.train  import make_loaders, run_training
from hmst.utils  import MODEL_SIZES, TRAIN_CFG, LOOKBACK, NUM_GRIDS
from hmst.utils  import calculate_metrics, build_forest_masks

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Load Data

In [ ]:
# Load the CHT population-flow parquet (shape: grids × hours)
DATA_PATH = 'data/2019_hourly_pop_byCHT.parquet'
df        = pd.read_parquet(DATA_PATH)
data      = df.values.astype('float32')   # (N, T)

print(f'Data shape: {data.shape}  →  {data.shape[0]} grids × {data.shape[1]} hours')

## 3. Choose a Model Size

All hyperparameters are looked up from a single dictionary.  
Changing `size_key` switches **every** model to that tier simultaneously.

In [ ]:
size_key = 'large'              # 'small' | 'base' | 'large'
size     = MODEL_SIZES[size_key]

print(f'--- {size_key.upper()} config ---')
for k, v in size.items():
    print(f'  {k:12s}: {v}')
print()
print('Training config (shared across all models):')
for k, v in TRAIN_CFG.items():
    print(f'  {k:12s}: {v}')

## 4. Build Data Loaders

In [ ]:
train_loader, val_loader, test_loader = make_loaders(
    data,
    lookback    = LOOKBACK,
    batch_size  = 32,
    train_frac  = 0.70,
    val_frac    = 0.85,
)
print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

## 5. Train a Baseline — AllGridLSTM

In [ ]:
lstm_model = AllGridLSTM(hidden_dim=size['hidden_dim'])

lstm_model, preds, trues, pe, te, t_secs, n_params = run_training(
    name         = f'All-Grid LSTM ({size_key})',
    model        = lstm_model,
    cfg          = TRAIN_CFG,
    train_loader = train_loader,
    val_loader   = val_loader,
    test_loader  = test_loader,
    device       = device,
)

mae, rmse, wmape, daily_dtw, step_dtw = calculate_metrics(te, pe)
print(f'\nLSTM ({size_key})  |  MAE={mae:.4f}  wMAPE={wmape:.2f}%  Params={n_params/1e3:.1f}K  Time={t_secs:.1f}s')

## 6. Train HMST-v2 (Proposed)

> **Note**: HMST requires pre-built `forest_masks` from the R-Tree construction step  
> (see Section 3 of the main research notebook `new_tree_building.ipynb`).

In [ ]:
# ── Placeholder: replace with actual forest_masks from R-Tree construction ──
# forest_masks = build_forest_masks(NUM_GRIDS, K_roots, root_labels, forest_levels)

# For demonstration, use identity-only masks (equivalent to full attention)
import numpy as np
dummy_mask    = np.zeros((NUM_GRIDS, NUM_GRIDS), dtype='float32')
forest_masks  = [dummy_mask] * 4

hmst_model = HMSTv2(
    num_grids    = NUM_GRIDS,
    lookback     = LOOKBACK,
    d_model      = size['d_model'],
    d_k          = size['d_k'],
    num_layers   = size['num_layers'],
    forest_masks = forest_masks,
)

hmst_model, preds, trues, pe, te, t_secs, n_params = run_training(
    name         = f'HMST-v2 ({size_key})',
    model        = hmst_model,
    cfg          = TRAIN_CFG,
    train_loader = train_loader,
    val_loader   = val_loader,
    test_loader  = test_loader,
    device       = device,
)

mae, rmse, wmape, daily_dtw, step_dtw = calculate_metrics(te, pe)
print(f'\nHMST-v2 ({size_key})  |  MAE={mae:.4f}  wMAPE={wmape:.2f}%  Params={n_params/1e3:.1f}K  Time={t_secs:.1f}s')

## 7. Compare Results

In [ ]:
# Collect results in a DataFrame — same pattern used in the research notebook
results = {
    f'All-Grid LSTM ({size_key})': {'MAE': mae, 'wMAPE (%)': wmape, 'Params (K)': n_params / 1e3},
    # add more models here following the same pattern ...
}

import pandas as pd
df_results = pd.DataFrame(results).T
print(df_results.to_markdown(floatfmt='.4f'))